# Lesson 01b · Make Q-learning generalize

The original table gives every exact 16-tile board a separate row. More data,
a different learning rate, and faster exploration decay are worth testing—but
none lets an update on one board directly help on another.

Here we keep **Q-learning's bootstrap target** and replace the dictionary with
**ten learned weights and ten engineered board features**. This is linear
function approximation, an explicit step beyond tabular Q-learning. There is
no neural network and no tree search. The original lesson remains unchanged.

Run this notebook from a fresh kernel. It demonstrates one update, reads the
completed comparison when present, and plays the selected checkpoint's saved
replay. It does not automatically repeat the full experiment suite.

In [ ]:
from pathlib import Path
import json
import inspect
import numpy as np
from IPython.display import Markdown, Image, display
import rl2048
from rl2048.game import Game2048, ACTION_NAMES
from rl2048.agents.feature_q import FEATURE_NAMES, FeatureQAgent, features, load_agent
from rl2048.view import replay_widget

ROOT = Path(rl2048.__file__).resolve().parent.parent
RESULTS = ROOT / "runs/variations"
print("Results:", RESULTS)

## What changed in the equation?

Previously, $Q(s,a)$ was one number in a dictionary row. Now:

$$Q_w(s,a)=w^T\phi(s,a) = \sum_{i=1}^{10} w_i\phi_i(s,a).$$

- $w$: ten **learned weights**, initialized to zero.
- $\phi(s,a)$: ten features describing the deterministic board after taking
  action $a$, before the random new tile appears.
- The features are engineered using the game rules; they are not discovered
  automatically. This supplies useful prior structure and generalization.

The target still uses the best legal next action:

$$y=\frac{r}{128}+\gamma\max_{a'\in A(s')}Q_w(s',a'),\qquad
\delta=y-Q_w(s,a).$$

At natural termination the next-state term is zero. At a time-limit truncation
it remains. We hold the target fixed when differentiating and update:

$$w\leftarrow w+\alpha\delta\phi(s,a).$$

This is **semi-gradient Q-learning**. One update can change estimated values
on many boards because they share weights. Function approximation plus an
off-policy bootstrap can also be unstable; the tabular convergence argument
does not automatically carry over.

Dividing every reward by 128 keeps the numbers manageable. It changes the
units of Q values, not the optimal policy for a fixed discount. Scores reported
in evaluation remain the original raw game scores. We also compare gamma and
learning-rate settings explicitly; the representation comparison is not an
ablation that holds every other parameter fixed.

## The ten features

| Feature | Meaning |
|---|---|
| Bias | Always 1; a shared offset |
| Empty fraction | Number of empty cells divided by 16 |
| Immediate merge reward / 128 | Reward computable from the known merge rules |
| Largest exponent / 16 | `log2(largest tile) / 16` |
| Largest tile in a corner | Whether at least one maximum tile occupies a corner |
| Horizontal order | How consistently rows increase or decrease |
| Vertical order | How consistently columns increase or decrease |
| Neighbor similarity | How close occupied neighboring tile exponents are |
| Equal-neighbor fraction | Number of equal occupied neighbor pairs divided by 24 |
| Corner-weighted tiles | Fraction of tile mass near the best corner orientation |

No weight is manually set to prefer a feature. An untrained agent has all-zero
Q values and selects randomly among legal ties. A non-learning greedy-merge
baseline separately checks how far knowing immediate rewards alone can get us.

In [ ]:
print(inspect.getsource(features))

## Follow a shared-weight update

Collect actual transitions until the first merge. Then inspect each feature
and its contribution to the change in weights. Some features of the next board
will resemble other boards, so these weights can help on states never seen
exactly before. Alpha is 0.001 and gamma 0.95, the selected validation setting.

In [ ]:
env = Game2048()
board, info = env.reset(seed=7)
agent = FeatureQAgent(alpha=0.001, gamma=0.95, seed=7)
for step in range(1000):
    action = agent.act(board, info["action_mask"], epsilon=1.0)
    next_board, reward, terminated, truncated, next_info = env.step(action)
    old_weights = agent.weights.copy()
    phi = features(board, action)
    trace = agent.update(board, action, reward, next_board, terminated, next_info["action_mask"])
    if reward > 0:
        break
    board, info = env.reset() if terminated or truncated else (next_board, next_info)
assert reward > 0
print("Action:", ACTION_NAMES[action], "Raw reward:", reward)
print("Scaled reward:", trace.reward, "Target:", trace.target, "TD error:", trace.td_error)
rows = ["| Feature | Value | Old weight | Weight change |", "|---|---:|---:|---:|"]
for name, value, old, new in zip(FEATURE_NAMES, phi, old_weights, agent.weights):
    rows.append(f"| {name} | {value:.4f} | {old:.5f} | {new-old:.6f} |")
display(Markdown("\n".join(rows)))
np.testing.assert_allclose(agent.weights, old_weights + agent.alpha * trace.td_error * phi)
print("Values on the different NEXT board:", agent.values(next_board))
env.close()

## How the additional files connect

```text
variations.py
  ├─ table settings → train.py → original QLearningAgent
  └─ feature settings → explicit loop → FeatureQAgent
                          ↓
                 checkpoint + metrics
                          ↓
             evaluate.py on VALIDATION games
                          ↓
          compare_variations.py picks one setting
                          ↓
             evaluate.py on fresh TEST games
                          ↓
             comparison.json + chart + replay
```

`agents/feature_q.py` contains features, the weight update, checkpoint loading,
and the non-learning greedy-merge control. `tests/test_feature_q.py` verifies
the mathematical update, generalization to another board, frozen evaluation,
and checkpoint/replay behavior. `view.py` and the HTML viewer work with either
checkpoint representation. `variations.md` records the settings and findings.

## Read the completed experiment

We used three training seeds (0, 1, 2) for every candidate. Each seed's agent
played the same 100 validation game seeds. We chose the setting with the highest
mean score across those three training seeds, then evaluated all three of its
agents on 200 **fresh test game seeds** each.

The same three table baselines were tested on those fresh games. Random,
untrained-feature and greedy-merge controls each play 200 games. Error bars are
standard deviations of the **three training-seed mean scores**, not confidence
intervals. The short feature run has a smaller budget and faster decay; it is
labelled separately rather than presented as an equal-budget comparison.

In [ ]:
report_path = RESULTS / "comparison.json"
report = json.loads(report_path.read_text()) if report_path.exists() else None
if report:
    for split in ["validation", "test"]:
        rows = [f"### {split.title()}", "", "| Setting | Mean score | Training-seed SD | Reach 1024 | Reach 2048 |", "|---|---:|---:|---:|---:|"]
        for name, row in report[split].items():
            sd = "—" if row["training_seed_sd"] is None else f"{row['training_seed_sd']:.0f}"
            rows.append(f"| {row['label']} | {row['mean_score']:.0f} | {sd} | {row['tile_reaching_rates']['1024']:.1%} | {row['tile_reaching_rates']['2048']:.1%} |")
        display(Markdown("\n".join(rows)))
    print("Selected setting:", report["winner"])
    display(Image(filename=str(RESULTS / "comparison.png")))
else:
    print("No saved comparison yet. The commands and settings are in variations.md.")

## Watch the selected agent

The demonstration uses the **middle-performing training seed on validation**,
and the same fixed demonstration game seed (2,000,000) used in lesson 01.
It is not the best game selected from many replays. The test metrics include
all three trained agents, regardless of which one is shown here.

In [ ]:
replay_path = RESULTS / "winner_replay.json"
if replay_path.exists():
    replay = json.loads(replay_path.read_text())
    print(replay["result"])
    display(replay_widget(replay["frames"]))
else:
    print("Train and compare the variants to generate the selected replay.")

## Inspect the learned weights

A weight's magnitude depends on feature scaling. A positive coefficient does
not prove a causal strategy: features are correlated, and decisions compare
the combined values of all legal actions. Inspect an individual board's feature
vectors alongside the weights to understand a decision.

In [ ]:
if report:
    checkpoint = RESULTS / f"{report['winner']}_s{report['demo_training_seed']}" / "checkpoint.npz"
    selected_agent = load_agent(checkpoint)
    if isinstance(selected_agent, FeatureQAgent):
        for name, weight in zip(FEATURE_NAMES, selected_agent.weights):
            print(f"{name:32s} {weight: .5f}")

## Run another feature experiment yourself

From the project root:

```bash
uv run python -m rl2048.variations --kind features --alpha 0.001 --gamma 0.95 --seed 0 --out runs/my_features
uv run python -m rl2048.evaluate --checkpoint runs/my_features/checkpoint.npz --seed-start 4000000 --episodes 200 --out runs/my_features/test
uv run python -m rl2048.view --checkpoint runs/my_features/checkpoint.npz --out runs/my_features/replay.html --open
```

If you use those test results to tune more settings, treat that set as validation
from then on and reserve fresh seeds for a later final test. Keep raw game score
as the comparison metric even though feature-Q values use scaled rewards.

Useful next questions: Why can a smaller learning rate help? Why are feature
choices useful prior knowledge? Why is this not tabular Q-learning anymore?
What would a neural network learn that these engineered features cannot express?